# 2. Drifter data cleaning

**What it does**
- Downloads the Clouddrift `gdp1h` hourly drifter arrays it needs (first run only) into `data/drifter/gdp1h/`, one variable at a time. This is the only notebook that downloads drifter data: notebooks 3 and 4 both load its extract.
- Opens those global arrays (about 197 million readings) without loading them into memory.
- Reads them 2 million readings at a time and keeps those in the study box (89°W–65°W, 49°S–4°S) and period (2001–2022).
- Converts SST from Kelvin to °C and drops non-finite values and anything outside 5 to 35 °C, keeps only readings with valid velocity, and drops readings south of 40°S and east of 71°W (Atlantic side of the Andes).
- Recovers each reading's drifter ID from `rowsize`, and saves the result once as a small regional extract, with velocity and each drifter's drogue-loss date for notebook 4. Later runs load the extract directly.

**Inputs**
- Clouddrift `gdp1h` (downloaded automatically on the first run; needs internet and about 9.5 GB of free disk)

**Outputs**
- `data/drifter/gdp1h/{lon,lat,time,sst,ve,vn,id,rowsize,drogue_lost_date}.npy`: the global arrays (not in git)
- `data/drifter/humboldt_sst_readings.npz`: about 2.5 million readings with id, time, lon, lat, SST in °C and velocity (`ve`, `vn`, m/s), plus each drifter's drogue-loss date (`drifter_id`, `drogue_lost_date`)
- `df`, one row per reading (shown at the end)

Run this notebook from the `notebooks/` folder, top to bottom. The first run downloads the data and takes a while; afterwards it takes seconds and peaks at about 1.5 GB of memory. To rebuild the extract (for example after changing the region), delete `humboldt_sst_readings.npz`. An older extract without the velocity and drogue arrays is rebuilt automatically.

## Download, load and filter the drifter readings

This keeps readings in 89°W–65°W, 49°S–4°S from 2001–2022 with SST between 5 and 35 °C and valid velocity, and drops readings south of 40°S and east of 71°W (those are in the Atlantic, not the Humboldt system). Notebooks 3 and 4 load this extract rather than applying the filters themselves. The global arrays are read from disk in chunks, which avoids loading all of them at once (more than 10 GB), and only the matching readings are saved to `data/drifter/humboldt_sst_readings.npz`. The extract is built once, and later runs load the small file.

Each array is downloaded in pieces (about 3.9 million readings each), three pieces at a time, with a progress bar. A piece that fails is retried automatically. Pieces are written straight to a temporary file that is renamed only when complete. If the download is interrupted, the next run downloads only the pieces still missing instead of starting again or leaving a broken file behind.

Separate `valid_velocity.npy` and `valid_sst_velocity.npy` mask files are not needed, because the SST and velocity filters are applied directly while reading.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

# All drifter data lives in its own folder: global gdp1h arrays in gdp1h/, the regional extract beside them
DRIFTER = Path("../data/drifter")
GDP1H = DRIFTER / "gdp1h"
EXTRACT = DRIFTER / "humboldt_sst_readings.npz"
GDP1H_VARS = ["lon", "lat", "time", "sst", "ve", "vn", "id", "rowsize", "drogue_lost_date"]
# Per-reading arrays in the extract, plus a per-drifter drogue table that notebook 4 uses
READING_KEYS = ["id", "time", "lon", "lat", "sst_degC", "ve", "vn"]
EXTRACT_KEYS = set(READING_KEYS) | {"drifter_id", "drogue_lost_date"}
pd.set_option("display.width", 200)

# Study time - 2001 to 2022
START, END = np.datetime64("2001-01-01", "ns"), np.datetime64("2023-01-01", "ns")

# Selected region - Humboldt Region, the same box as notebook 4
R = dict(lon=(-89, -65), lat=(-49, -4))
SST_RANGE = (5, 35)  # °C; same as notebook 4. Real winter SST south of 40S falls below 10 C, so lower values are sensor errors


def download_gdp1h(variables, piece_chunks=10, workers=3, retries=5):
    """Download any missing Clouddrift gdp1h arrays into data/drifter/gdp1h, one variable at a time.

    Each array is fetched in pieces of `piece_chunks` remote chunks, `workers` pieces at a time, and
    written straight to disk with a progress bar. A piece that fails (the server sometimes drops the
    connection) is retried up to `retries` times. An interrupted download resumes with only the
    missing pieces. More than 3 workers tends to make the server disconnect.
    """
    missing = [v for v in variables if not (GDP1H / f"{v}.npy").exists()]
    if not missing:
        return
    import time
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from clouddrift.datasets import gdp1h  # opens the dataset remotely; nothing is read until .values
    from tqdm.auto import tqdm
    GDP1H.mkdir(parents=True, exist_ok=True)
    ds = gdp1h()

    for v in missing:
        var = ds[v]
        n = var.shape[0]
        step = (var.encoding.get("chunks") or (n,))[0] * piece_chunks
        partial = GDP1H / f"{v}.partial.npy"
        log_file = GDP1H / f"{v}.partial.pieces"  # start index of each piece already on disk, one per line

        if partial.exists() and log_file.exists():
            done = {int(x) for x in log_file.read_text().split()}
            out = np.load(partial, mmap_mode="r+")
        else:
            done = set()
            out = np.lib.format.open_memmap(partial, mode="w+", dtype=var.dtype, shape=(n,))
            log_file.write_text("")
        todo = [a for a in range(0, n, step) if a not in done]

        def fetch(a):
            b = min(a + step, n)
            for attempt in range(retries + 1):
                try:
                    return a, b, var[a:b].values
                except Exception:
                    if attempt == retries:
                        raise
                    time.sleep(2 * (attempt + 1))  # back off, then try the piece again

        bar = tqdm(total=n, initial=n - sum(min(a + step, n) - a for a in todo),
                   desc=f"gdp1h {v}", unit="val", unit_scale=True)
        pool = ThreadPoolExecutor(workers)
        try:
            with open(log_file, "a") as log:
                for future in as_completed([pool.submit(fetch, a) for a in todo]):
                    a, b, values = future.result()
                    out[a:b] = values
                    out.flush()
                    log.write(f"{a}\n")  # recorded only after the piece is safely on disk
                    log.flush()
                    bar.update(b - a)
        finally:
            pool.shutdown(wait=False, cancel_futures=True)  # on an interrupt, don't wait for queued pieces
            bar.close()

        del out
        partial.rename(GDP1H / f"{v}.npy")  # only a finished download gets the real name
        log_file.unlink()
        print(f"  saved {GDP1H / f'{v}.npy'}")


def build_extract(chunk=2_000_000):
    """Keep valid SST and velocity readings in the study region and period, reading the global arrays in chunks."""
    raw = {v: np.load(GDP1H / f"{v}.npy", mmap_mode="r") for v in ["lon", "lat", "time", "sst", "ve", "vn"]}
    ids = np.load(GDP1H / "id.npy")
    last_row = np.cumsum(np.load(GDP1H / "rowsize.npy"))
    parts = {k: [] for k in READING_KEYS}

    for a in range(0, len(raw["lon"]), chunk):
        b = a + chunk
        lon, lat = np.asarray(raw["lon"][a:b]), np.asarray(raw["lat"][a:b])
        in_R = ((lon >= R["lon"][0]) & (lon <= R["lon"][1])
                & (lat >= R["lat"][0]) & (lat <= R["lat"][1]))
        rows = np.flatnonzero(in_R)
        if len(rows) == 0:
            continue

        t = np.asarray(raw["time"][a:b])[rows]
        # Convert SST from Kelvin to Celsius
        sst_c = np.asarray(raw["sst"][a:b])[rows] - 273.15
        ve, vn = (np.asarray(raw[v][a:b])[rows] for v in ["ve", "vn"])
        valid = ((t >= START) & (t < END) & np.isfinite(sst_c)
                 & (sst_c >= SST_RANGE[0]) & (sst_c <= SST_RANGE[1])
                 & np.isfinite(ve) & np.isfinite(vn)
                 # Drop readings east of the Andes: these are in the Atlantic, not the Humboldt system
                 & ~((lat[rows] < -40) & (lon[rows] > -71)))
        rows = rows[valid]

        parts["id"].append(ids[np.searchsorted(last_row, a + rows, side="right")])
        parts["time"].append(t[valid])
        parts["lon"].append(lon[rows])
        parts["lat"].append(lat[rows])
        parts["sst_degC"].append(sst_c[valid])
        parts["ve"].append(ve[valid])
        parts["vn"].append(vn[valid])

    partial = DRIFTER / "humboldt_sst_readings.partial.npz"
    np.savez(partial, **{k: np.concatenate(v) for k, v in parts.items()},
             drifter_id=ids, drogue_lost_date=np.load(GDP1H / "drogue_lost_date.npy"))
    partial.rename(EXTRACT)


def extract_is_current():
    """True if the extract exists and has every array notebooks 3 and 4 need."""
    if not EXTRACT.exists():
        return False
    with np.load(EXTRACT) as f:
        return EXTRACT_KEYS <= set(f.files)


if not extract_is_current():
    download_gdp1h(GDP1H_VARS)
    print("Building the regional extract ...", flush=True)
    build_extract()

obs = np.load(EXTRACT)
df = pd.DataFrame({k: obs[k] for k in READING_KEYS})

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month
df["doy"] = df["time"].dt.dayofyear

print(f"Observations with SST and velocity: {len(df):,} from {df['id'].nunique():,} drifters")


Observations with SST and velocity: 2,537,111 from 659 drifters


/var/folders/j8/wthqn_1x6lb6ltm2m0rkfgrm0000gn/T/ipykernel_13410/488506369.py:148: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df["doy"] = df["time"].dt.dayofyear
